# E1 · The income in euros that wipes out the amount/income ratio

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Understand why a regularized logistic changes its predictions if you change the **unit of measure** of a column (euros versus thousands of euros), find out which coefficient gets sacrificed and fix it with
a scaler inside the pipeline. Verify that after the fix the model is **invariant** to the units.

## Dataset

`ds.loans(with_missing=False)` with `ratio = amount/income` (scale about 0.5), `age`, `job_tenure` and `income` (in euros, scale about 30,000): stratified 70/30 split. `evaluate(model, k)` (given) trains a copy
of the model dividing `income` by `k` and returns the test AUC.

## Educational purpose

Recognize a bug that gives no errors or warnings: `C=0.1` penalizes all the coefficients the same way, but a column with an enormous scale needs a tiny coefficient and is in fact penalized
very little, while a column with a small scale (the ratio) gets squashed. The result changes with the unit of measure: it is a model that depends on an arbitrary choice.

## Task

1. Run the colleague's cell: `auc_bug` (income in euros) and `auc_bug_k_euro` (income in thousands of euros). The same model on the same data should give the same result: save `unit_difference = auc_bug_k_euro − auc_bug`.
2. Train the colleague's model on the training set (income in euros) and save `bug_ratio_coef` (coefficient of `ratio`).
3. Fix it: `ok_model` = `StandardScaler` + `LogisticRegression(C=0.1, max_iter=1000)`. Save `auc_ok` and `auc_ok_k_euro` (with `evaluate`) and `invariance = abs(auc_ok − auc_ok_k_euro)`.
4. Train `ok_model` on the training set and save `ok_ratio_coef`: the coefficient of `ratio` **in original units** (coefficient of the model divided by the standard deviation used by the scaler: `scaler.scale_[0]`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]
cols = ["ratio", "age", "job_tenure", "income"]
X_tr, X_te, y_tr, y_te = train_test_split(df[cols], df["defaulted"], test_size=0.3, stratify=df["defaulted"], random_state=0)


def evaluate(model, k=1.0):
    """Test AUC of a copy of the model, with `income` divided by k (k = 1000: thousands of euros)."""
    a, b = X_tr.copy(), X_te.copy()
    a["income"] /= k
    b["income"] /= k
    return roc_auc_score(y_te, clone(model).fit(a, y_tr).predict_proba(b)[:, 1])

## Your work

### The colleague's code

In [ ]:
bug_model = LogisticRegression(C=0.1, max_iter=1000)
auc_bug = evaluate(bug_model)                 # income in euros
auc_bug_k_euro = evaluate(bug_model, 1000)    # same data, income in thousands of euros
print(f"test AUC: {auc_bug:.3f} with the income in euros | {auc_bug_k_euro:.3f} in thousands of euros")

### Your diagnosis and the fix

In [ ]:
unit_difference = None
bug_ratio_coef = None

In [ ]:
ok_model = None
auc_ok = None
auc_ok_k_euro = None
invariance = None
ok_ratio_coef = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint="`ok_model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000))`; `ok_model.fit(X_tr, y_tr)`; `ok_model[-1].coef_[0][0] / ok_model[0].scale_[0]`. The colleague's coefficient: `bug_model.fit(X_tr, y_tr).coef_[0][0]` (on the copy, not on the one used by `evaluate`)."):
    check(auc_bug < 0.75, "the colleague's model with the income in euros is poor (AUC below 0.75)", "don't modify the colleague's code")
    check(close(unit_difference, auc_bug_k_euro - auc_bug) and unit_difference > 0.04, "changing only the unit of measure the AUC changes by over 0.04", "auc_bug_k_euro - auc_bug")
    check(bug_ratio_coef is not None and abs(bug_ratio_coef) < 0.02, "in the colleague's model the coefficient of the ratio has been squashed (below 0.02)", "coef_[0][0] of the model trained on the training set in euros")
    check(invariance is not None and invariance < 1e-3 and auc_ok > 0.77, "with the scaler the AUC does not depend on the unit of measure (gap below 0.001) and is 0.77 or more", "StandardScaler before LogisticRegression")
    check(ok_ratio_coef > 50 * abs(bug_ratio_coef), "in original units the right coefficient of the ratio is hundreds of times larger than the colleague's", "coef of the model / scaler.scale_[0]")
done()

---
The commented solution is in `solutions/E_find_the_bug/E1_regularization_without_scaling.ipynb`. Open it after you pass the check, or after you have really tried.